In [ ]:
from mass_function import MassFunction

# original source: https://github.com/reineking/pyds/blob/master/pyds.py


In [ ]:
ALPHA = 0.3
# 説明では$a$とした。

## 使い方

ライブラリは一般的（３つ以上も値を取れる。）な集合に対して作成してある。

（集合、確信度、集合要素）
を指定して定義する。

In [ ]:
mass_function_1 = MassFunction(source=[({"exist"}, ALPHA)],
                                       coreset={"exist", "notexist"})
mass_function_2 = MassFunction(source=[({"exist"}, ALPHA)],
                                       coreset={"exist", "notexist"})

## 確信度の表示

Pythonの集合(set)ではなく必ず変更できない集合（frozenset）で配列の値を取得する必要がある。

In [ ]:
mass_function_1

In [ ]:
print(
mass_function_1[frozenset({"exist"})], 
mass_function_1[frozenset({"notexist"})],
mass_function_1[frozenset({"exist","notexist"})],
)
# 確かめ
print(ALPHA,0, 1-ALPHA)

## 結合

In [ ]:
combined_mass_function1 = mass_function_1.combine(mass_function_2)


## 結果取得



In [ ]:
print(
combined_mass_function1[frozenset({"exist"})], 
combined_mass_function1[frozenset({"notexist"})],
combined_mass_function1[frozenset({"exist","notexist"})],
)
# 確かめ
print(2*ALPHA-ALPHA**2, 0, (1-ALPHA)**2)

In [ ]:
print(
combined_mass_function1[frozenset({"exist"})], 
combined_mass_function1[frozenset({"notexist"})],
combined_mass_function1[frozenset({"exist","notexist"})],
)
# 確かめ
print(2*ALPHA-ALPHA**2, 0, (1-ALPHA)**2)

In [ ]:
mass_function_3 = MassFunction(source=[({"notexist"}, ALPHA)],
                                       coreset={"exist", "notexist"})

In [ ]:
combined_mass_function2 = mass_function_1.combine(mass_function_3)

In [ ]:
print(
combined_mass_function2[frozenset({"exist"})], 
combined_mass_function2[frozenset({"notexist"})],
combined_mass_function2[frozenset({"exist","notexist"})],
)
# 確かめ
print(ALPHA/(1+ALPHA), ALPHA/(1+ALPHA), (1-ALPHA)/(1+ALPHA))

# 繰り返し時の振る舞い

### 有、が続く場合

In [ ]:
ALPHA = 0.3
NITER = 10
mass_function = MassFunction(source=[({"exist"}, ALPHA)],
                                       coreset={"exist", "notexist"})
print(mass_function)
m_history = []
for iter in range(NITER):
    mass_function_add = MassFunction(source=[({"exist"}, ALPHA)],
                                     coreset={"exist", "notexist"})
    mass_function = mass_function.combine(mass_function_add)
    m_exist = mass_function[frozenset({"exist"})]
    m_notexist = mass_function[frozenset({"notexist"})]
    m_unknown = mass_function[frozenset({"exist","notexist"})]
    m_history.append([m_exist, m_notexist, m_unknown])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df_m_history = pd.DataFrame(m_history, columns=["exist", "notexist", "unknown"])
fig, ax = plt.subplots()
df_m_history.plot(y="exist", marker="o", ax=ax)
df_m_history.plot(y="unknown", marker="o", ax=ax)


# ランダムな場合

In [ ]:
import numpy as np

# np.random.seed(seed=1)

ALPHA = 0.3
R_THRESHOLD = 0.5
NITER = 10
mass_function = MassFunction(source=[({"exist"}, ALPHA)],
                                       coreset={"exist", "notexist"})
print(mass_function)
m_history = []
for iter in range(NITER-1): # 全部でNITER個用いる。
    r = np.random.random(size=1) # [0,1)を与える。
    r_int = np.random.randint(low=0, high=3, size=1)
    print(iter,r, r_int)
    if r_int==0:
        mass_function_add = MassFunction(source=[({"exist"}, ALPHA)],
                                         coreset={"exist", "notexist"})
    elif r_int==1:
        mass_function_add = MassFunction(source=[({"notexist"}, ALPHA)],
                                         coreset={"exist", "notexist"})
    else:
        mass_function_add = MassFunction(source=[({"notexist","exist"}, 1)],
                                         coreset={"exist", "notexist"})
        
    mass_function = mass_function.combine(mass_function_add)
    m_exist = mass_function[frozenset({"exist"})]
    m_notexist = mass_function[frozenset({"notexist"})]
    m_unknown = mass_function[frozenset({"exist","notexist"})]
    m_history.append([r_int, m_exist, m_notexist, m_unknown])
df_m_history = pd.DataFrame(m_history, columns=["flag", "exist", "notexist", "unknown"])
print("# of exist", np.count_nonzero(df_m_history["flag"].values)+1)
fig, ax = plt.subplots()
df_m_history.plot(y="exist", marker="o", ax=ax)
df_m_history.plot(y="notexist", marker="o", ax=ax)
df_m_history.plot(y="unknown", marker="o", ax=ax)

それぞれ、ALPHAやNITERを変えて見てください。



### 結果について

ALPHAの値によっては、
想像より早く0もしくは１に変化します。

人間の感覚に合わせるならばALPHAを調整してください。
新帰納法としては観測データに対して妥当な推論が可能なようにALPHAを調整します。
